# Tool Calling Demo

This notebook demonstrates **tool calling** (OpenAI also calls it *function calling*) with a hand-written tool definition and a simple simulated weather tool. It uses OpenAI's **Chat Completions API**, the same API as the prompt engineering demo.

## Learning objectives

By the end of this workshop, you should understand:

1. How to create a normal Python function.
2. How to describe that function to an LLM as a tool definition, with its arguments described in JSON Schema.
3. How the LLM selects a tool and generates arguments.
4. How the Python application executes the function.
5. How the tool result is returned to the LLM for the final answer.

## 1. Setup

In [1]:
!pip install -q openai

### Add your OpenAI API key as a Colab secret (one time only)

1. Click the **key icon** ("Secrets") in the left sidebar.
2. Click **+ Add new secret**.
3. **Name:** `OPENAI_API_KEY` (exactly this, in capitals). **Value:** your OpenAI API key.
4. Turn on **Notebook access** for this notebook.

If you are not running in Colab, or the secret can't be read, the next cell asks you to type the key instead.

> Never paste your API key directly into a code cell: it would be saved with the notebook and visible to anyone you share it with.

In [2]:
import os
import json
from openai import OpenAI

def load_openai_key():
    """Read the OpenAI API key from Colab Secrets; ask for it only if that fails."""
    try:
        from google.colab import userdata
        return userdata.get("OPENAI_API_KEY")
    except ImportError:
        print("Not running in Colab.")
    except Exception as e:
        # e.g. the secret is missing, or Notebook access is turned off
        print(f"Could not read the Colab secret OPENAI_API_KEY ({type(e).__name__}).")
    from getpass import getpass
    return getpass("Enter your OpenAI API key: ")

os.environ["OPENAI_API_KEY"] = load_openai_key()
client = OpenAI()

# A fast, low-cost non-reasoning model is enough for this demo.
MODEL = "gpt-4o-mini"
print("OpenAI client is ready. Model:", MODEL)

OpenAI client is ready. Model: gpt-4o-mini


## 2. Create the actual Python function

For this workshop, we use **simulated weather data** so that the focus stays on tool calling rather than external APIs.

At this stage, `get_weather()` is just a normal Python function. The LLM does not know it exists yet.

In [3]:
def get_weather(location):
    """Simulated weather function for workshop purposes."""

    weather_data = {
        "Singapore": {"temperature": 31, "condition": "Partly cloudy", "humidity": 75},
        "London": {"temperature": 18, "condition": "Rainy", "humidity": 80},
        "Tokyo": {"temperature": 27, "condition": "Sunny", "humidity": 65},
    }

    if location in weather_data:
        return {"location": location, **weather_data[location]}

    return {
        "location": location,
        "error": "Weather data is not available for this location.",
    }

### Test the Python function directly

No LLM is involved in this test.

In [4]:
print(get_weather("Singapore"))

{'location': 'Singapore', 'temperature': 31, 'condition': 'Partly cloudy', 'humidity': 75}


## 3. Define the tool manually

The tool definition tells the model **what the tool is**, but not how it works. It has two parts:

- **Tool settings:** `name` (what the model calls), `description` (when to use it; the model decides from this text alone, so write it like a prompt) and `strict` (the arguments must match the schema exactly).
- **`parameters`:** a **JSON Schema** describing the arguments: their names, types and which are required. Strict mode needs every argument in `required` and `"additionalProperties": False`.

The definition does **not** contain the Python code. The model only ever sees this description.

In Chat Completions, the settings are nested inside a `"function"` object. (OpenAI's newer Responses API uses the same keys without that nesting.)

In [5]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a specified city. "
                           "Use this when the user asks about the weather right now.",
            "strict": True,
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city name, for example Singapore, London, or Tokyo.",
                    }
                },
                "required": ["location"],
                "additionalProperties": False,
            },
        },
    }
]

# Maps each tool name the model may request to the Python function that runs it.
# To add a tool, add its definition to `tools` and its function here.
available_functions = {
    "get_weather": get_weather,
}

## 4. The agent loop

`run_agent()` repeats the tool-calling loop:

1. Send the conversation and the tool definitions to the model.
2. If the reply has **no tool calls**, it is the final answer: stop.
3. Otherwise, run each requested function and send its result back as a `"tool"` message, then go back to step 1.

`MAX_TURNS` is a **stopping condition**: it ends the loop even if the model keeps asking for tools. Every agent needs one.

In [6]:
MAX_TURNS = 5


def run_agent(user_question):
    messages = [{"role": "user", "content": user_question}]

    for turn in range(MAX_TURNS):
        # Call the LLM
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
        )
        message = response.choices[0].message

        # Keep the model's reply in the conversation (including any tool calls)
        messages.append(message)

        # -----------------------------------
        # No tool call -> final answer
        # -----------------------------------
        if not message.tool_calls:
            print("\nFinal answer:")
            print(message.content)
            return message.content

        # -----------------------------------
        # Execute the requested tools
        # -----------------------------------
        for tool_call in message.tool_calls:
            name = tool_call.function.name
            print("\nLLM requested tool:", name)
            print("Arguments:", tool_call.function.arguments)

            # The arguments arrive as a JSON string: convert to a Python dictionary
            arguments = json.loads(tool_call.function.arguments)

            function = available_functions.get(name)
            if function is None:
                # Unknown tool: still send a result, so the model can recover
                result = {"error": f"Unknown tool: {name}"}
            else:
                result = function(**arguments)

            print("Tool result:", result)

            # Send the tool result back to the LLM, linked to this tool call by its id
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(result),
                }
            )

    print(f"\nStopped after {MAX_TURNS} turns without a final answer.")
    return None

## 5. Run it

In [13]:
run_agent("What's the weather like in Singapore?")


LLM requested tool: get_weather
Arguments: {"location":"Singapore"}
Tool result: {'location': 'Singapore', 'temperature': 31, 'condition': 'Partly cloudy', 'humidity': 75}

Final answer:
The current weather in Singapore is partly cloudy with a temperature of 31°C and a humidity level of 75%.


'The current weather in Singapore is partly cloudy with a temperature of 31°C and a humidity level of 75%.'

## 6. Exercises

Try these with `run_agent(...)` in a new cell:

1. Ask for the weather in `London`, then in `Tokyo`.
2. Ask for a city that is not in `weather_data`, such as `Paris`. How does the model use the error result?
3. Ask a question that does **not** need the tool, such as *"What is the capital of Japan?"*. Does the model call `get_weather`?
4. Ask about two cities in one question: *"Compare the weather in London and Tokyo."* How many tool calls does the model make?